Generating arrays for h5 file

In [1]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.audio_transforms as at
import src.custom_modules as cm
import src.spatial_attn_lightning as binaural_lightning
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [2]:
# path to 376 wiki words
swc_path = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'

# path o common voice words not included in training
# df = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/commonvoice_9_en/manifest_all_word_alignments.pdpkl')

In [3]:
manifest = pd.read_pickle(swc_path + 'manifest.pdpkl')

In [4]:
manifest

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,raw_clip_dur_in_s,raw_clip_end_in_s,raw_clip_start_in_s,raw_src_fn,raw_total_file_duration_in_s,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,a-c-norman,3.0,3.0,0.0,swc,0.32,2094.94,2094.62,/scratch2/weka/mcdermott/msaddler/swc/english/...,2175.444172,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,above
1,jebjoya,3.0,3.0,0.0,swc,0.49,1715.87,1715.38,/scratch2/weka/mcdermott/msaddler/swc/english/...,2793.356190,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,according
2,caninedoubletake,3.0,3.0,0.0,swc,0.36,169.03,168.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,987.438730,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,across
3,karltalk,3.0,3.0,0.0,swc,0.60,2429.51,2428.91,/scratch2/weka/mcdermott/msaddler/swc/english/...,4802.892336,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,action
4,s-whistler,3.0,3.0,0.0,swc,0.80,1149.87,1149.07,/scratch2/weka/mcdermott/msaddler/swc/english/...,4463.715556,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,activities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,incledon,3.0,3.0,0.0,swc,0.69,231.72,231.03,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.573152,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,world
372,ama1016,3.0,3.0,0.0,swc,0.42,193.47,193.05,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.502041,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,writing
373,zanimum,3.0,3.0,0.0,swc,0.27,13.92,13.65,/scratch2/weka/mcdermott/msaddler/swc/english/...,452.154921,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,written
374,tonyle,3.0,3.0,0.0,swc,0.31,1208.98,1208.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,2359.540680,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,wrote


In [5]:
fn_pkl_src = '/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl'
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'

In [6]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))
words = list(word_dict.keys())
words = [word.replace("'", "") for word in words]
manifest_all_words = pd.read_pickle(fn_pkl_dst)
# filter out words not in 'words' list
manifest_all_words = manifest_all_words[manifest_all_words['word'].isin(words)]
word_counts = manifest_all_words['word'].value_counts()

In [7]:
down_df = manifest_all_words.groupby(['word', 'gender']).sample(1, replace=False)

In [8]:
final_df = down_df.groupby('word').filter(lambda x: len(x) == 2)

In [9]:
final_df = final_df.reset_index().rename(columns={'index':'src_ix'})

In [10]:
final_df

,src_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,119262,popularoutcast,0.37,1748.11,1747.74,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2316.145488,about
1,842116,peppage,0.31,641.36,641.05,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,703.377415,about
2,379636,laurahale,0.37,152.63,152.26,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,204.859501,above
3,240338,tonyle,0.30,211.78,211.48,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,585.608707,above
4,246904,ama1016,0.92,166.39,165.47,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1181.505306,access
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1573,653426,donbert,0.35,1554.46,1554.11,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1597.876825,yellow
1574,119867,popularoutcast,0.32,338.67,338.35,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2373.870295,young
1575,926218,jasonlamarche,0.22,1057.61,1057.39,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3476.793469,young
1576,378993,laurahale,0.30,61.52,61.22,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,166.938413,younger


In [11]:
all_words_not_filtered = pd.read_pickle(fn_pkl_dst)
oov_words = all_words_not_filtered[~all_words_not_filtered['word'].isin(words)]

In [12]:
talkers = final_df['client_id'].unique()
oov_words[oov_words['client_id'].isin(talkers)].client_id.value_counts()

client_id
s-whistler               60197
matthewdgonzalez         49291
mangst                   36135
the-voice-of-hassocks    35187
alexkillby               26728
                         ...  
rebelguys2                  49
ralmin                      31
eleshamc                    21
richard-taylor              20
ixthus                       8
Name: count, Length: 219, dtype: int64

In [13]:
samples_per_talker = {talker:count for talker,count in final_df.client_id.value_counts().items()}
viables_cues = oov_words[oov_words.client_id.isin(talkers)]

cues = viables_cues.groupby('client_id').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='client_id', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'cue_src_ix'}, inplace=True)

In [14]:
final_df.sort_values(by='client_id', inplace=True)
final_df.reset_index(inplace=True, drop=True)


cues.sort_values(by='client_id', inplace=True)
cues.reset_index(inplace=True, drop=True)



### Merge cues with foregrounds  
cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']] = cues[['word', 'cue_src_ix', 'client_id', 'src_fn', 'clip_start_in_s', 'clip_end_in_s']]
# Combine as experiment dataframe
training_df = final_df.join(cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']])
assert (training_df['client_id'] == training_df['cue_client_id']).all(), "Cue and Target talkers don't match!"

In [15]:
training_df

,src_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word,cue_word,cue_src_ix,cue_client_id,cue_fn,cue_start_in_s,cue_end_in_s
0,684104,0x0077be,0.15,315.39,315.24,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,775.033333,working,feet,684020,0x0077be,/scratch2/weka/mcdermott/msaddler/swc/english/...,256.11,256.41
1,992570,1904-cc,0.69,740.17,739.48,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,popular,assembly,992565,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,580.77,581.23
2,992919,1904-cc,0.22,1875.29,1875.07,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,lives,desktop,992705,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1090.33,1090.84
3,992531,1904-cc,0.58,515.73,515.15,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,television,processing,993015,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1981.07,1981.75
4,992467,1904-cc,0.37,451.56,451.19,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,video,tightly,992413,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,369.97,370.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1573,103779,zanimum,0.65,2578.31,2577.66,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3221.953016,experience,well,101553,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,117.28,117.57
1574,105180,zanimum,0.37,503.71,503.34,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,617.248798,reading,hoax,96501,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,314.11,314.59
1575,102074,zanimum,0.75,672.01,671.26,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3221.953016,performance,tate,100695,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,88.36,88.82
1576,563773,zegoma-beach,0.33,260.24,259.91,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,479.647347,lower,that,563892,zegoma-beach,/scratch2/weka/mcdermott/msaddler/swc/english/...,335.68,335.83


In [16]:
training_df = pd.read_pickle('/om2/user/rphess/Auditory-Attention/binaural_test_manifest.pdpkl')

Trial run of Testing (if this works, will spatialize the manifest and write test script)

In [17]:
def pad_or_trim_to_len(x, n, mode='both', kwargs_pad={}):
    """
    Increases or decreases the length of a one-dimensional signal
    by either padding or triming the array. If the difference
    between `len(x)` and `n` is odd, this function will default to
    adding/removing the extra sample at the end of the signal.
    
    Args
    ----
    x (np.ndarray): one-dimensional input signal
    n (int): length of output signal
    mode (str): specify which end of signal to modify
        (default behavior is to symmetrically modify both ends)
    kwargs_pad (dict): keyword arguments for np.pad function
    
    Returns
    -------
    x_out (np.ndarray): one-dimensional signal with length `n`
    """
    assert len(np.array(x).shape) == 1, "input must be 1D array"
    assert mode.lower() in ['both', 'start', 'end']
    n_diff = np.abs(len(x) - n)
    if len(x) > n:
        if mode.lower() == 'end':
            x_out = x[:n]
        elif mode.lower() == 'start':
            x_out = x[-n:]
        else:
            x_out = x[int(np.floor(n_diff / 2)):-int(np.ceil(n_diff / 2))]
    elif len(x) < n:
        if mode.lower() == 'end':
            pad_width = [0, n_diff]
        elif mode.lower() == 'start':
            pad_width = [n_diff, 0]
        else:
            pad_width = [int(np.floor(n_diff / 2)), int(np.ceil(n_diff / 2))]
        kwargs = {'mode': 'constant'}
        kwargs.update(kwargs_pad)
        x_out = np.pad(x, pad_width, **kwargs)
    else:
        x_out = x
    assert len(x_out) == n
    return x_out

In [18]:
def get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0):
    """
    This function loads an audio file and excerpts a clip with the specified
    duration. Target durations that exceed clip boundaries are handled with
    zero-padding (applied to all signals but sliced away when not needed).
    This function also handles resampling (via soxr) and re-scaling.
    """
    jitter_in_s = 0
    jitter_via_zero_padding = True
    if dfi.clip_dur_in_s > dur:
        # Take a random segment if clip duration is longer than excerpt
        clip_start_in_s = np.random.uniform(
            low=dfi.clip_start_in_s,
            high=dfi.clip_start_in_s + dfi.clip_dur_in_s - dur,
            size=None)
        clip_end_in_s = clip_start_in_s + dur
        jitter_via_zero_padding = False
    else:
        # Temporally jitter clip by extending either start or end time
        jitter_in_s = np.random.uniform(
            low=-dfi.clip_dur_in_s * jitter_fraction,
            high=dfi.clip_dur_in_s * jitter_fraction,
            size=None)
        if pad_with_context:
            # If using context, adjust clip start and end times to account for jitter and context
            if jitter_in_s > 0:
                clip_start_in_s = dfi.clip_start_in_s - (2 * np.abs(jitter_in_s))
                clip_end_in_s = dfi.clip_end_in_s
            else:
                clip_start_in_s = dfi.clip_start_in_s
                clip_end_in_s = dfi.clip_end_in_s + (2 * np.abs(jitter_in_s))
            clip_dur_in_s = clip_end_in_s - clip_start_in_s
            jitter_via_zero_padding = False
            context_pad_in_s = (dur - clip_dur_in_s) / 2
        else:
            clip_start_in_s = dfi.clip_start_in_s
            clip_end_in_s = dfi.clip_end_in_s
            context_pad_in_s = 0
        clip_start_in_s = clip_start_in_s - context_pad_in_s
        clip_end_in_s = clip_end_in_s + context_pad_in_s
    clip_dur_in_s = clip_end_in_s - clip_start_in_s
    # Load audio, pad, slice with indexes that account for padding
    load_full_file = True
    if (clip_start_in_s >= 0) and (clip_end_in_s < dfi.total_file_duration_in_s):
        # Attempt to read only the specified excerpt
        myfile = sf.SoundFile(dfi.src_fn)
        if myfile.seekable():
            src_sr = myfile.samplerate
            frame_start = int(np.round(clip_start_in_s * src_sr))
            frames = int(np.round(clip_dur_in_s * src_sr))
            myfile.seek(frame_start)
            y = myfile.read(frames, always_2d=True)
            y = np.mean(y, axis=1)
            load_full_file = False
    if load_full_file:
        # If impossible, read full audio file
        y, src_sr = sf.read(dfi.src_fn, always_2d=True)
        y = np.mean(y, axis=1)
        frame_start = int(np.round(clip_start_in_s * src_sr))
        frames = int(np.round(clip_dur_in_s * src_sr))
        if frame_start < 0:
            y = np.pad(y, [-frame_start, 0])
            frame_start = 0
        if frame_start + frames > len(y):
            y = np.pad(y, [0, frame_start + frames - len(y)])
        y = y[frame_start : frame_start + frames]
    # Resample from src_sr to sr
    y = soxr.resample(y, src_sr, sr).astype(np.float32)
    # If not yet jittered, apply jitter at end via asymmetric zero-padding
    if jitter_via_zero_padding:
        jitter_pad_width = int(np.round(2 * np.abs(jitter_in_s) * sr))
        if jitter_in_s > 0:
            y = np.pad(y, [jitter_pad_width, 0]).astype(np.float32)
        else:
            y = np.pad(y, [0, jitter_pad_width]).astype(np.float32)
    # Zero-pad or trim to length (fixes off by one errors)
    y = pad_or_trim_to_len(y, int(dur * sr))
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y

In [19]:
df = training_df.sample(350)

In [20]:
foregrounds = []
cues = []
for i, row in df.iterrows():
    src_ix = row['src_ix']
    cue_ix = row['cue_src_ix']
    src_row = all_words_not_filtered.iloc[src_ix]
    cue_row = all_words_not_filtered.iloc[cue_ix]
    foregrounds.append(get_excerpt(src_row))
    cues.append(get_excerpt(cue_row))

In [21]:
df['loaded_foreground'] = foregrounds

In [22]:
df['loaded_cue'] = cues

In [23]:
mdl_ckpt = 'exp/word_task_voice_and_loc_cue_v04/checkpoints/epoch=7-step=73184.ckpt'

In [24]:
config = yaml.load(open('config/binaural_attn/word_task_voice_and_loc_cue_v04.yml', 'r'), Loader=yaml.FullLoader)

In [25]:
# config

In [26]:
config['noise_kwargs']['low_snr'] = "clean"
config['noise_kwargs']['high_snr'] = "clean"

In [27]:
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config).cuda()

num_classes={'num_words': 800}
Model performing word task


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [28]:
print("Loading speaker array room BRIRs")
list_data_dict = []
for elev in [-20, -10, 0, 10, 20, 30, 40]:
    for azim in np.arange(0, 360, 5):
        data_dict = {
            'azim': azim,
            'elev': elev,
            'brir': [],
        }
        for ear in ['l', 'r']:
            basename = f'{elev}elev_{azim}az_2.47x2.60y2.00z_{ear}.wav'
            if elev >= 0:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/', basename)
            else:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/neg_elevs/', basename)
            assert os.path.exists(fn)
            brir, sr_src = sf.read(fn)
            sr = 50000
            brir = soxr.resample(brir.astype(np.float32), sr_src, sr)
            data_dict['brir'].append(brir)
        data_dict['sr'] = sr
        data_dict['brir'] = np.stack(data_dict['brir']).T
        list_data_dict.append(data_dict)
df_brir = pd.DataFrame(list_data_dict)
df_brir_room = df_brir[np.logical_and.reduce([
    df_brir['azim'] % 10 == 0,
    ~(np.logical_and(df_brir['azim'] > 90, df_brir['azim'] < 270)),
    df_brir['elev'] >= 0,
])].reset_index()
print(f"Loaded speaker array room BRIRs ({len(df_brir_room)})")

Loading speaker array room BRIRs
Loaded speaker array room BRIRs (95)


In [29]:
brir_00 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 0) & (df_brir_room['elev'] == 0)]['brir'].values[0])
brir_900 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 270) & (df_brir_room['elev'] == 0)]['brir'].values[0])
brir_neg900 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 90) & (df_brir_room['elev'] == 0)]['brir'].values[0])

In [30]:
brir_00.shape

torch.Size([25000, 2])

In [31]:
brir_00 = torch.flip(brir_00, dims=[0])
brir_900 = torch.flip(brir_900, dims=[0])
brir_neg900 = torch.flip(brir_neg900, dims=[0])

In [32]:

audio_transforms = model.audio_transforms
coch_transforms = model.coch_gram.cuda()

In [33]:
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
class_map = {k.replace("'", ''):v for k,v in class_map.items()}

In [34]:
model.config['num_workers'] = 0
model.config['hparas']['batch_size'] = 32

In [35]:
def mass_spatialize(words, ir):
    """Uses pytorch to convolve all sounds in words with 2 channel IR given in ir"""
    n_words = words.shape[0]
    words_padded = [torch.nn.functional.pad(word, (ir.shape[0] - 1, 0)) for word in words]
    ir = ir.T.unsqueeze(1)
    words_padded = torch.stack(words_padded)
    spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1).cuda(), ir.cuda()).cuda()
    return spatialized

In [36]:
# make torch.nn.Module version of spatilaize
class Spatialize(torch.nn.Module):
    def __init__(self, ir):
        super(Spatialize, self).__init__()
        ir = torch.flip(torch.from_numpy(ir), dims=[0])
        self.n_taps = ir.shape[0]
        ir = ir.T.unsqueeze(1)
        self.register_buffer("ir", ir)

    def forward(self, words):
        n_words = words.shape[0]
        # pad last dim of words with ir.shape[0] - 1 zeros
        words_padded = torch.nn.functional.pad(words, (self.n_taps - 1, 0))
        spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1), self.ir)
        # resize to desired shape
        spatialized = spatialized[:, :, 12500:137500]
        return spatialized


## Compare perf using updated eval logic



In [40]:


#put both at 0,0 

centered_ir = Spatialize(df_brir_room[(df_brir_room['azim'] == 0) & (df_brir_room['elev'] == 0)]['brir'].values[0]).cuda()
right_ir = Spatialize(df_brir_room[(df_brir_room['azim'] == 270) & (df_brir_room['elev'] == 0)]['brir'].values[0]).cuda()

config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config).cuda()
audio_transforms = model.audio_transforms.cuda()
coch_transforms = model.coch_gram.cuda()


results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]

    cue = right_ir(cue.cuda())
    fg = right_ir(fg.cuda())
    distractor_signal = right_ir(distractor_signal.cuda())

    cue, _ = audio_transforms(cue, None)
    mixture, _ = audio_transforms(fg, distractor_signal)

    cue, mixture = coch_transforms(cue, mixture)

    logits = model(cue, mixture, None)

    result = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach().numpy().astype('int')

    results.append(label == result)
    confusions.append(distractor_label == result)




print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")


num_classes={'num_words': 800}
Model performing word task


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


  1%|          | 3/350 [00:06<09:18,  1.61s/it]/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/_inductor/cudagraph_trees.py:2049: UserWarning: Unable to hit fast path of CUDAGraphs because of pending, uninvoked backwards. Consider running with torch.no_grad() or using torch._inductor.cudagraph_mark_step_begin() before each model invocation
  warnings.warn(
100%|██████████| 350/350 [01:06<00:00,  5.25it/s]

Accuracy = 0.20857142857142857
Confusions = 0.08285714285714285


## Make sure spatialization modulel actually works as intened 

In [76]:
fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)

fg_old = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()[:, :, 12500:137500]

ir = df_brir_room[(df_brir_room['azim'] == 0) & (df_brir_room['elev'] == 0)]['brir'].values[0]
ir = torch.flip(torch.from_numpy(ir), dims=[0])

print((ir == brir_00).all())

fg_padded = torch.nn.functional.pad(fg, (ir.shape[0] - 1, 0))
fg_manual =  torch.nn.functional.conv1d(fg_padded.view(1, 1, -1), ir.T.unsqueeze(1))
fg_manual = fg_manual[:, :, 12500:137500]

print((fg_manual == fg_old).all())

# spatialized 
centered_ir = Spatialize(df_brir_room[(df_brir_room['azim'] == 0) & (df_brir_room['elev'] == 0)]['brir'].values[0]).cuda()
print((centered_ir.ir.cpu() == ir.T.unsqueeze(1)).all())
fg_nn = centered_ir(fg.cuda()).cpu()
print((fg_nn == fg_old).all())

In [77]:
fg_old.shape, fg_nn.shape

(torch.Size([1, 2, 125000]), torch.Size([1, 2, 112502]))

## Test1 : Clean signals 

In [67]:
### No Spatialization no distractor

In [72]:
# No spatialization


results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    
    cue = np.repeat(row['loaded_cue'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    fg = np.repeat(row['loaded_foreground'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    bg = np.repeat(distractor['loaded_foreground'].item()[np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    # cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    # fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    # bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, None)[0]
    cue, scene = coch_transforms(cue.cuda(), scene.cuda())
    
    out = model.forward(cue, scene, None)

    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")


  0%|          | 0/350 [00:00<?, ?it/s]

/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,
/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/_inductor/compile_fx.py:135: U

Accuracy = 0.7571428571428571
Confusions = 0.0


In [73]:
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")


Accuracy = 0.7571428571428571
Confusions = 0.0


## With distractor no spatialization

In [36]:
## Try with distractor, no spatialize 

# No spatialization

config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config).cuda()
audio_transforms = model.audio_transforms
coch_transforms = model.coch_gram.cuda()

results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    
    cue = np.repeat(row['loaded_cue'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    fg = np.repeat(row['loaded_foreground'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    bg = np.repeat(distractor['loaded_foreground'].item()[np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    # cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    # fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    # bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0]
    cue, scene = coch_transforms(cue.cuda(), scene.cuda())
    out = model.forward(cue, scene, None)
    
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)


print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")


num_classes={'num_words': 800}
Model performing word task


/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


  0%|          | 0/350 [00:00<?, ?it/s]/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/_inductor/compile_fx.py:135: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
  0%|          | 1/350 [00:10<1:02:11, 10.69s/it]/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/_inductor/cudagraph_trees.py:2049: UserWarning: Unable to hit fast path of CUDAGraphs because of pending, uninvoked backwards. Consider running with torch.no_grad() or using torch._inductor.cudagraph_mark_step_begin() before each model invocation
  warnings.warn(
100%|██████████| 350/350 [00:39<00:00,  8.84it/s]

Accuracy = 0.3171428571428571
Confusions = 0.12857142857142856


Case 1: target at 0, distractor at 90
Case 2: target at -90, distractor at 90
Case 3: both at 0

In [69]:
rev_map = {v:k for k,v in class_map.items()}

## case 1 target at 0, distractor at right 90 


In [94]:
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config).cuda()
audio_transforms = model.audio_transforms

model = model.eval()

num_classes={'num_words': 800}
Model performing word task
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


In [82]:
# case 1 target at 0, distractor at right 90 

results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0]

    cue, scene = coch_transforms(cue.cuda(), scene.cuda())

    out = model.forward(cue, scene, None)
    
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)

    
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")


  1%|          | 3/350 [00:06<09:12,  1.59s/it]/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/_inductor/cudagraph_trees.py:2049: UserWarning: Unable to hit fast path of CUDAGraphs because of pending, uninvoked backwards. Consider running with torch.no_grad() or using torch._inductor.cudagraph_mark_step_begin() before each model invocation
  warnings.warn(
100%|██████████| 350/350 [01:08<00:00,  5.13it/s]

Accuracy = 0.5057142857142857
Confusions = 0.005714285714285714


Last checkpoint    
Accuracy = 0.66   
Confusions = 0.0  


In [96]:
# case 1 target at 0, distractor at left 90 

results = []
confusions = []
# with torch.no_grad():
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0]


    cue, scene = coch_transforms(cue.cuda(), scene.cuda())

    out = model.forward(cue, scene, None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)

    
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")


  1%|          | 3/350 [00:07<11:00,  1.90s/it]/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_tf/lib/python3.11/site-packages/torch/_inductor/cudagraph_trees.py:2049: UserWarning: Unable to hit fast path of CUDAGraphs because of pending, uninvoked backwards. Consider running with torch.no_grad() or using torch._inductor.cudagraph_mark_step_begin() before each model invocation
  warnings.warn(
 43%|████▎     | 150/350 [01:28<01:57,  1.70it/s]


KeyboardInterrupt: 

## case 2 - target left 90 distractor right 90 


In [88]:
# case 2 - target left 90 distractor right 90 

results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_neg900.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    fg = mass_spatialize(fg.cuda(), brir_neg900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0]


    cue, scene = coch_transforms(cue.cuda(), scene.cuda())

    out = model.forward(cue, scene, None)

    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

  0%|          | 0/350 [00:00<?, ?it/s]

100%|██████████| 350/350 [01:48<00:00,  3.24it/s]

Accuracy = 0.6628571428571428
Confusions = 0.0


last ckpt    
Accuracy = 0.62    
Confusions = 0.0

## case 2 - target right 90 distractor left 90 


In [89]:
# case 2 - target right 90 distractor left 90 

results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_900.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0]


    cue, scene = coch_transforms(cue.cuda(), scene.cuda())

    out = model.forward(cue, scene, None)

    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

  0%|          | 0/350 [00:00<?, ?it/s]

100%|██████████| 350/350 [02:05<00:00,  2.79it/s]

Accuracy = 0.5342857142857143
Confusions = 0.0


In [ ]:
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

Accuracy = 0.45714285714285713
Confusions = 0.008571428571428572


old ckpt     
Accuracy = 0.46285714285714286   
Confusions = 0.002857142857142857

In [90]:
# target at right 90, distractor at 0
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_900.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_00.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])

    scene = audio_transforms(fg, bg)[0]


    cue, scene = coch_transforms(cue.cuda(), scene.cuda())

    out = model.forward(cue, scene, None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

  0%|          | 0/350 [00:00<?, ?it/s]

100%|██████████| 350/350 [02:23<00:00,  2.44it/s]

Accuracy = 0.4942857142857143
Confusions = 0.0


In [91]:
# see if we can put cue at different location 
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]
    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0]

    cue, scene = coch_transforms(cue.cuda(), scene.cuda())

    out = model.forward(cue, scene, None)

    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)

    
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

  0%|          | 0/350 [00:00<?, ?it/s]

100%|██████████| 350/350 [02:40<00:00,  2.19it/s]

Accuracy = 0.06571428571428571
Confusions = 0.05142857142857143


In [92]:
# Put cue at distractor location 
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_neg900.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0]

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0]


    cue, scene = coch_transforms(cue.cuda(), scene.cuda())

    out = model.forward(cue, scene, None)

    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

100%|██████████| 350/350 [02:58<00:00,  1.96it/s]

Accuracy = 0.002857142857142857
Confusions = 0.5028571428571429


In [ ]:
torch.Tensor([1, 2, 3]).max(-1)

torch.return_types.max(
values=tensor(3.),
indices=tensor(2))

In [ ]:
cond_dict = pickle.load(open("/om2/user/rphess/Auditory-Attention/speaker_room_0_elev_conditions.pkl", 'rb'))

In [ ]:
import scipy